# Figure 5 — pages
Assembles the panels written by `psccm_figures.ipynb` into the figure pages:
- **Figure 5** — a schematic | b colocalization + expression; c all-pathway scatter | d cell types; e | f core-cluster
  heatmaps (same height)
- **Figure 5Sa** — a replicate design | b replicate pairs; c all cardiomyocyte sets; d all stemness sets
- **Figure 5Sb** — 60-pathway panel (Hallmark + custom): a colocalized | b differentially expressed

Panels are nested as vector SVG (text stays editable). Each page is laid out on A4 width and cropped tight to its
content → editable SVG in `../../illustrator-figures/`, PDF/PNG in `figures/`. Needs `cairosvg` for the PDF/PNG.

In [1]:
import copy
import io
import xml.etree.ElementTree as ET

import cairosvg
import numpy as np
from PIL import Image

FIG = "figures"
PAGES = "../../illustrator-figures"                     # page SVGs live with the other Illustrator files
SVG_NS = "http://www.w3.org/2000/svg"
XLINK_NS = "http://www.w3.org/1999/xlink"
for prefix, uri in [("", SVG_NS), ("xlink", XLINK_NS), ("rdf", "http://www.w3.org/1999/02/22-rdf-syntax-ns#"),
                    ("cc", "http://creativecommons.org/ns#"), ("dc", "http://purl.org/dc/elements/1.1/")]:
    ET.register_namespace(prefix, uri)                  # keep matplotlib's prefixes when re-serializing
PAGE_W, PAGE_H = 595.276, 841.89                       # A4 (pt) layout frame; pages are cropped afterwards
CONTENT_X, CONTENT_W = 28, 543                         # panel area left edge / width
GAP_Y = 22                                             # vertical gap between rows (pt)
PAD = 4                                                # margin kept around the content when cropping (pt)


def load_svg(name):
    """Root element of a panel SVG in figures/."""
    return ET.parse(f"{FIG}/{name}.svg").getroot()


def size(root):
    """(width, height) of an SVG from its viewBox."""
    return tuple(float(v) for v in root.get("viewBox").split()[2:])


def new_page():
    """Blank A4 page with an opaque white background."""
    page = ET.Element(f"{{{SVG_NS}}}svg", width=f"{PAGE_W}pt", height=f"{PAGE_H}pt", viewBox=f"0 0 {PAGE_W} {PAGE_H}")
    ET.SubElement(page, f"{{{SVG_NS}}}rect", id="background", x="0", y="0", width=f"{PAGE_W}", height=f"{PAGE_H}",
                  fill="#ffffff")
    return page


def nest(page, src_root, x, y, scale):
    """Place a panel SVG on the page at (x, y), scaled. Returns its bottom."""
    vb = tuple(float(v) for v in src_root.get("viewBox").split())
    child = ET.SubElement(page, f"{{{SVG_NS}}}svg", x=f"{x}", y=f"{y}", width=f"{vb[2] * scale}",
                          height=f"{vb[3] * scale}", viewBox=" ".join(f"{v}" for v in vb),
                          preserveAspectRatio="xMinYMin meet", overflow="hidden")
    for el in src_root:
        child.append(copy.deepcopy(el))
    return y + vb[3] * scale


def letter(page, text, x, y):
    """Panel letter (Arial 12, lowercase + period)."""
    t = ET.SubElement(page, f"{{{SVG_NS}}}text", x=f"{x}", y=f"{y}",
                      style="font-family: Arial, ArialMT, sans-serif; font-size: 12px; fill: #000000")
    t.text = f"{text}."


def row(page, specs, y, gap=14):
    """Place panels side by side at a common height spanning CONTENT_W; specs = [(letter, panel name)]. Returns the bottom."""
    roots = [load_svg(name) for _, name in specs]
    sizes = [size(r) for r in roots]
    height = (CONTENT_W - gap * (len(specs) - 1)) / sum(w / h for w, h in sizes)
    x = CONTENT_X
    for (name, _), root, (w, h) in zip(specs, roots, sizes):
        nest(page, root, x, y, height / h)
        letter(page, name, x - (11 if x == CONTENT_X else 4), y + 8)
        x += w * height / h + gap
    return y + height


def full_width(page, name, panel, y, max_scale=None):
    """One panel spanning the content width (optionally capped in scale), left-aligned. Returns its bottom."""
    svg = load_svg(panel)
    w, _ = size(svg)
    scale = CONTENT_W / w if max_scale is None else min(CONTENT_W / w, max_scale)
    letter(page, name, CONTENT_X - 11, y + 8)
    return nest(page, svg, CONTENT_X, y, scale)


def write_cropped(page, name):
    """Crop the page to its drawn content (+ PAD); save the SVG to PAGES and the PDF/PNG to figures/."""
    # find the non-white extent on a high-resolution render
    png = cairosvg.svg2png(bytestring=ET.tostring(page), scale=4)
    ink = np.asarray(Image.open(io.BytesIO(png)).convert("L")) < 250
    px = ink.shape[1] / PAGE_W                          # pixels per pt
    rows = np.where(ink.any(axis=1))[0]
    cols = np.where(ink.any(axis=0))[0]
    x0, x1 = max(cols[0] / px - PAD, 0), min((cols[-1] + 1) / px + PAD, PAGE_W)
    y0, y1 = max(rows[0] / px - PAD, 0), min((rows[-1] + 1) / px + PAD, PAGE_H)
    w = x1 - x0
    h = y1 - y0
    page.set("viewBox", f"{x0:.2f} {y0:.2f} {w:.2f} {h:.2f}")
    page.set("width", f"{w:.2f}pt")
    page.set("height", f"{h:.2f}pt")
    background = page.find(f"{{{SVG_NS}}}rect[@id='background']")
    background.attrib.update(x=f"{x0:.2f}", y=f"{y0:.2f}", width=f"{w:.2f}", height=f"{h:.2f}")
    ET.ElementTree(page).write(f"{PAGES}/{name}.svg", xml_declaration=True, encoding="utf-8")
    cairosvg.svg2pdf(url=f"{PAGES}/{name}.svg", write_to=f"{FIG}/{name}.pdf")
    cairosvg.svg2png(url=f"{PAGES}/{name}.svg", write_to=f"{FIG}/{name}.png", scale=2)
    print(f"{name}: {w:.0f} x {h:.0f} pt")

## Figure 5
Row 1: b at the size it would have if a and b shared one height, right-aligned; the schematic (a) grows into the remaining
width and b is centred vertically on it. Row 2: c | d. Row 3: e | f core-cluster heatmaps at the same height.

In [2]:
page = new_page()
a_svg = load_svg("fig5a_schematic")
b_svg = load_svg("fig5b_coloc_expression")
a_w, a_h = size(a_svg)
b_w, b_h = size(b_svg)
b_scale = (CONTENT_W - 24) / (a_w / a_h + b_w / b_h) / b_h
b_x = CONTENT_X + CONTENT_W - b_w * b_scale
a_scale = (b_x - CONTENT_X - 4) / a_w
a_bottom = nest(page, a_svg, CONTENT_X, 17, a_scale)
b_bottom = nest(page, b_svg, b_x, 17 + (a_h * a_scale - b_h * b_scale) / 2, b_scale)   # centred on a
bottom = max(a_bottom, b_bottom)
letter(page, "a", CONTENT_X - 11, 25)
letter(page, "b", b_x + 4, 25)
bottom = row(page, [("c", "fig5c_coloc_vs_expression"), ("d", "fig5d_celltype_specificity")], bottom + GAP_Y)
row(page, [("e", "fig5e_sarcomere_core_cluster"), ("f", "fig5f_cardiomyocyte_core_cluster")], bottom + GAP_Y)
write_cropped(page, "Figure5")

Figure5: 558 x 723 pt


## Supplementary Figure 5Sa (robustness)
Replicate design (a) | replicate pairs (b); all cardiomyocyte (c) and stemness (d) sets.

In [3]:
page = new_page()
bottom = row(page, [("a", "fig5Sa_a_replicate_design"), ("b", "fig5Sa_b_replicate_sets")], 17)
bottom = full_width(page, "c", "fig5Sa_c_all_cardiac_sets", bottom + GAP_Y, max_scale=0.58)
full_width(page, "d", "fig5Sa_d_all_stemness_sets", bottom + GAP_Y, max_scale=0.58)
write_cropped(page, "Figure5Sa")

Figure5Sa: 558 x 780 pt


## Supplementary Figure 5Sb (pathway context)
Colocalized (a) | differentially expressed (b) pathways of the 60-pathway panel, one shared scale.

In [4]:
page = new_page()
a_svg = load_svg("fig5Sb_a_pathway_colocalization")
b_svg = load_svg("fig5Sb_b_pathway_expression")
a_w, _ = size(a_svg)
b_w, _ = size(b_svg)
scale = (CONTENT_W - 18) / (a_w + b_w)
b_x = CONTENT_X + a_w * scale + 18
nest(page, a_svg, CONTENT_X, 17, scale)
nest(page, b_svg, b_x, 17, scale)
letter(page, "a", CONTENT_X - 11, 25)
letter(page, "b", b_x - 4, 25)
write_cropped(page, "Figure5Sb")

Figure5Sb: 558 x 415 pt
